# SQL CTEs and Recursive Queries — Complete Reference

| Pattern | Use Case |
|---------|----------|
| Simple CTE | Named subquery, reuse in same query |
| Multiple CTEs | Pipeline of transformations |
| Recursive CTE | Hierarchy traversal, sequence generation |
| CTE vs subquery | Readability, reuse, optimization |
| CTE pipeline | Multi-step data pipeline in one query |

**Mental model**: A CTE (`WITH name AS (...)`) is a named subquery scoped to a single statement. Recursive CTEs add a self-referencing UNION to traverse graphs, hierarchies, or generate series.

```sql
WITH
  cte1 AS (SELECT ...),
  cte2 AS (SELECT ... FROM cte1)
SELECT ... FROM cte2;

-- Recursive:
WITH RECURSIVE cte(col) AS (
  SELECT seed           -- anchor
  UNION ALL
  SELECT next FROM cte  -- recursive step
  WHERE stop_condition
)
SELECT * FROM cte;
```

## Visual Model

```
SIMPLE CTE — named subquery
───────────────────────────
  WITH dept_avg AS (
      SELECT dept, AVG(salary) avg_sal FROM emp GROUP BY dept
  )
  SELECT e.name, e.salary, d.avg_sal
  FROM emp e JOIN dept_avg d ON e.dept = d.dept
  WHERE e.salary > d.avg_sal;

  ┌──────────┐     ┌────────────┐
  │ dept_avg │────►│ final query│
  └──────────┘     └────────────┘

RECURSIVE CTE — hierarchy traversal
────────────────────────────────────
  Org chart: CEO → VPs → Directors → Managers

  employees: id, name, manager_id
  1  Alice  NULL    ← anchor (manager_id IS NULL)
  2  Bob    1
  3  Carol  1
  4  Dave   2
  5  Eve    2

  Iteration 0 (anchor):  Alice
  Iteration 1:           Bob, Carol    (manager_id IN {1})
  Iteration 2:           Dave, Eve     (manager_id IN {2})
  ⤷ UNION ALL accumulates; terminates when no new rows added

MULTIPLE CTEs — transformation pipeline
─────────────────────────────────────────
  raw → cleaned → aggregated → ranked → filtered
  each CTE is one step; final SELECT outputs result
```

## Setup — Libraries and Config

In [ ]:
import sqlite3

print(f"sqlite3 version: {sqlite3.sqlite_version}")

def make_db():
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.executescript("""
        -- Org hierarchy
        CREATE TABLE employees (
            id INTEGER PRIMARY KEY,
            name TEXT,
            manager_id INTEGER REFERENCES employees(id),
            department TEXT,
            salary REAL,
            hire_date TEXT
        );
        INSERT INTO employees VALUES
            (1,'Alice',NULL,'Exec',200000,'2015-01-01'),
            (2,'Bob',1,'Eng',150000,'2017-03-15'),
            (3,'Carol',1,'Sales',140000,'2017-06-01'),
            (4,'Dave',2,'Eng',120000,'2018-07-20'),
            (5,'Eve',2,'Eng',115000,'2019-02-10'),
            (6,'Frank',3,'Sales',100000,'2019-09-01'),
            (7,'Grace',3,'Sales',95000,'2020-01-15'),
            (8,'Heidi',4,'Eng',90000,'2021-06-01'),
            (9,'Ivan',4,'Eng',88000,'2022-03-01'),
            (10,'Judy',5,'Eng',85000,'2022-08-15');

        -- Orders
        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY,
            customer_id INTEGER,
            order_date TEXT,
            amount REAL,
            status TEXT
        );
        INSERT INTO orders VALUES
            (1,101,'2024-01-05',250,'completed'),
            (2,101,'2024-01-20',180,'completed'),
            (3,102,'2024-01-08',500,'completed'),
            (4,103,'2024-01-12',75,'cancelled'),
            (5,101,'2024-02-01',320,'completed'),
            (6,102,'2024-02-14',200,'completed'),
            (7,104,'2024-02-20',1200,'completed'),
            (8,103,'2024-03-05',90,'completed'),
            (9,104,'2024-03-10',800,'completed'),
            (10,102,'2024-03-25',150,'cancelled');
    """)
    conn.commit()
    return conn

def run(conn, sql, title=""):
    cur = conn.execute(sql)
    rows = cur.fetchall()
    if not rows:
        print(f"{title}: (no rows)")
        return []
    cols = [d[0] for d in cur.description]
    col_w = [max(len(c), max(len(str(r[c])) for r in rows)) for c in cols]
    sep = "  ".join("-" * w for w in col_w)
    header = "  ".join(c.ljust(w) for c, w in zip(cols, col_w))
    if title:
        print(f"\n=== {title} ===")
    print(header)
    print(sep)
    for r in rows:
        print("  ".join(str(r[c]).ljust(w) for c, w in zip(cols, col_w)))
    return rows

conn = make_db()
print("Database ready.")

## Decision Map — CTE Patterns

```
When to use a CTE?
│
├─ Subquery used more than once in same query?  → CTE (avoid repeating)
├─ Deeply nested subquery hard to read?         → CTE pipeline
├─ Traversing parent-child hierarchy?           → RECURSIVE CTE
├─ Generating a number/date sequence?           → RECURSIVE CTE
├─ Intermediate aggregation needed?             → CTE → join back to base
└─ Debugging complex query step by step?        → CTE (comment out outer steps)

CTE vs Subquery:
  CTE:      readable, reusable within same statement, debuggable
  Subquery: inline, not reusable, can be harder to read when nested
  Both:     optimizer often treats them identically (materializes or not)

CTE vs Temp Table:
  CTE:       scoped to single query, no disk I/O (usually)
  Temp Table: persists for session, can be indexed, better for multi-step scripts

Recursive CTE structure:
  WITH RECURSIVE name(cols) AS (
    anchor_query           -- seeds the recursion
    UNION ALL              -- UNION removes dups; UNION ALL keeps all (faster)
    recursive_step         -- references name, adds rows each iteration
  )
  SELECT * FROM name;
  -- Terminates: when recursive step returns 0 rows
  -- Guard: always include a depth limit or stop condition to prevent infinite loops
```

## Pattern 1 — Simple CTE (Named Subquery)

In [ ]:
# Without CTE: nested subquery — hard to read, subquery runs twice if referenced twice
# With CTE: named, defined once, referenced multiple times

# Find employees earning above their department average
run(conn, """
    WITH dept_avg AS (
        SELECT department, AVG(salary) AS avg_salary
        FROM employees
        GROUP BY department
    )
    SELECT
        e.name,
        e.department,
        e.salary,
        ROUND(d.avg_salary, 0) AS dept_avg,
        ROUND(e.salary - d.avg_salary, 0) AS above_avg_by
    FROM employees e
    JOIN dept_avg d ON e.department = d.department
    WHERE e.salary > d.avg_salary
    ORDER BY e.department, e.salary DESC
""", "Employees above dept average")

# CTE referenced twice: once for the filter, once for % calculation
run(conn, """
    WITH dept_stats AS (
        SELECT department, AVG(salary) avg_sal, MAX(salary) max_sal
        FROM employees
        GROUP BY department
    )
    SELECT
        e.name, e.department, e.salary,
        ROUND(100.0 * e.salary / d.max_sal, 1) AS pct_of_max
    FROM employees e
    JOIN dept_stats d ON e.department = d.department
    ORDER BY e.department, pct_of_max DESC
""", "Salary as pct of dept max")

## Pattern 2 — Multiple CTEs (Transformation Pipeline)

In [ ]:
# Chain multiple CTEs: each step builds on the previous
# This is the SQL equivalent of a Pandas pipeline

# Pipeline: raw orders → completed only → customer totals → ranked → top customers
run(conn, """
    WITH
    -- Step 1: filter to completed orders
    completed_orders AS (
        SELECT * FROM orders WHERE status = 'completed'
    ),
    -- Step 2: aggregate per customer
    customer_totals AS (
        SELECT
            customer_id,
            COUNT(*)    AS order_count,
            SUM(amount) AS total_spend,
            MIN(order_date) AS first_order,
            MAX(order_date) AS last_order
        FROM completed_orders
        GROUP BY customer_id
    ),
    -- Step 3: rank by total spend
    ranked AS (
        SELECT
            *,
            RANK() OVER (ORDER BY total_spend DESC) AS spend_rank
        FROM customer_totals
    )
    -- Final: show all with rank
    SELECT * FROM ranked ORDER BY spend_rank
""", "Multi-CTE pipeline: orders → customer summary → ranked")

## Pattern 3 — Recursive CTE (Hierarchy Traversal)

In [ ]:
# Traverse org chart from top down using RECURSIVE CTE
# anchor: root node (manager_id IS NULL)
# recursive step: join to find direct reports
# depth column: tracks level in hierarchy
# path column: tracks ancestry chain for display

run(conn, """
    WITH RECURSIVE org_tree AS (
        -- Anchor: start with root (no manager)
        SELECT
            id, name, manager_id, department, salary,
            0 AS depth,
            name AS path
        FROM employees
        WHERE manager_id IS NULL

        UNION ALL

        -- Recursive: find direct reports of each node
        SELECT
            e.id, e.name, e.manager_id, e.department, e.salary,
            t.depth + 1,
            t.path || ' > ' || e.name
        FROM employees e
        JOIN org_tree t ON e.manager_id = t.id
        WHERE t.depth < 5  -- safety guard against cycles
    )
    SELECT
        SUBSTR('        ', 1, depth * 2) || name AS org_chart,
        department,
        salary,
        depth,
        path
    FROM org_tree
    ORDER BY path
""", "Org chart traversal (recursive CTE)")

# Aggregate: total salary under each manager
run(conn, """
    WITH RECURSIVE subordinates(manager_id, subordinate_id) AS (
        SELECT id, id FROM employees  -- each person is their own subordinate
        UNION ALL
        SELECT s.manager_id, e.id
        FROM employees e
        JOIN subordinates s ON e.manager_id = s.subordinate_id
    )
    SELECT
        m.name AS manager,
        COUNT(DISTINCT s.subordinate_id) - 1 AS headcount,
        SUM(e.salary) AS total_team_salary
    FROM subordinates s
    JOIN employees m ON s.manager_id = m.id
    JOIN employees e ON s.subordinate_id = e.id
    WHERE s.manager_id != s.subordinate_id
    GROUP BY m.name
    ORDER BY total_team_salary DESC
    LIMIT 5
""", "Team salary under each manager")

## Pattern 4 — CTE vs Subquery

In [ ]:
# Compare the same query written as nested subquery vs CTE
# Goal: customers whose total spend exceeds overall average spend

print("=== Subquery version (nested, hard to read) ===")
subquery_sql = """
    SELECT customer_id, total_spend
    FROM (
        SELECT customer_id, SUM(amount) AS total_spend
        FROM orders
        WHERE status = 'completed'
        GROUP BY customer_id
    )
    WHERE total_spend > (
        SELECT AVG(cust_total)
        FROM (
            SELECT SUM(amount) AS cust_total
            FROM orders
            WHERE status = 'completed'
            GROUP BY customer_id
        )
    )
    ORDER BY total_spend DESC
"""
run(conn, subquery_sql, "Subquery version")

print("\n=== CTE version (readable pipeline) ===")
cte_sql = """
    WITH
    customer_totals AS (
        SELECT customer_id, SUM(amount) AS total_spend
        FROM orders
        WHERE status = 'completed'
        GROUP BY customer_id
    ),
    avg_spend AS (
        SELECT AVG(total_spend) AS avg_val FROM customer_totals
    )
    SELECT c.customer_id, c.total_spend, ROUND(a.avg_val, 2) AS avg_spend
    FROM customer_totals c, avg_spend a
    WHERE c.total_spend > a.avg_val
    ORDER BY c.total_spend DESC
"""
run(conn, cte_sql, "CTE version")

print("""
Key differences:
  CTE: each step named and readable; avg_spend CTE computed once, reused
  Subquery: nested 3 levels deep; avg subquery would run for every row in some engines
  Performance: most optimizers materialize CTEs similarly — readability is the main win
""")

## Pattern 5 — CTE for Sequence / Date Generation

In [ ]:
# Recursive CTE to generate integer sequences and date ranges
# Critical for: filling in missing dates, generating test data, calendar joins

# Generate integers 1..10
run(conn, """
    WITH RECURSIVE nums(n) AS (
        SELECT 1
        UNION ALL
        SELECT n + 1 FROM nums WHERE n < 10
    )
    SELECT n FROM nums
""", "Integer sequence 1..10")

# Generate date range for Jan 2024
run(conn, """
    WITH RECURSIVE dates(d) AS (
        SELECT DATE('2024-01-01')
        UNION ALL
        SELECT DATE(d, '+1 day') FROM dates WHERE d < DATE('2024-01-07')
    )
    SELECT d AS date FROM dates
""", "Date series Jan 1-7 2024")

# Fill in missing days: left join calendar to orders
run(conn, """
    WITH RECURSIVE calendar(d) AS (
        SELECT '2024-01-01'
        UNION ALL
        SELECT DATE(d, '+1 day') FROM calendar WHERE d < '2024-01-10'
    ),
    daily_orders AS (
        SELECT order_date, COUNT(*) AS num_orders, SUM(amount) AS daily_revenue
        FROM orders
        WHERE order_date BETWEEN '2024-01-01' AND '2024-01-10'
        GROUP BY order_date
    )
    SELECT
        c.d AS date,
        COALESCE(d.num_orders, 0) AS num_orders,
        COALESCE(d.daily_revenue, 0) AS revenue
    FROM calendar c
    LEFT JOIN daily_orders d ON c.d = d.order_date
    ORDER BY c.d
""", "Fill missing dates with 0 revenue")

## Full Decision Map

```
CTE USAGE GUIDE
───────────────
Simple CTE         → complex subquery used once for readability
Multi-step CTE     → chain of transformations; each step debuggable
CTE + Window func  → compute window over aggregated CTE result
Recursive CTE      → parent-child hierarchy, graph traversal, sequences
CTE + LEFT JOIN    → fill gaps (calendar, sequence) in sparse data

RECURSIVE CTE TEMPLATE
──────────────────────
WITH RECURSIVE cte(id, parent, depth, path) AS (
  -- ANCHOR: base case
  SELECT id, parent, 0, CAST(id AS TEXT)
  FROM table WHERE parent IS NULL

  UNION ALL

  -- RECURSIVE STEP: join to extend tree
  SELECT t.id, t.parent, c.depth + 1, c.path || '>' || t.id
  FROM table t
  JOIN cte c ON t.parent = c.id
  WHERE c.depth < 10   -- safety limit!
)
SELECT * FROM cte;

PITFALLS
────────
• Forgetting depth limit → infinite loop on cyclic data
• UNION (not UNION ALL) in recursive step → dedup kills performance
• Non-recursive CTEs may be inlined by optimizer (no materialization)
• SQLite: RECURSIVE keyword required even for non-recursive CTEs in some versions
• Correlated subquery in CTE = still runs per-row; move to JOIN instead
```

## Cheat Sheet

```sql
-- Basic CTE
WITH cte AS (SELECT ...) SELECT ... FROM cte;

-- Multiple CTEs
WITH
  a AS (SELECT ... FROM raw),
  b AS (SELECT ... FROM a),
  c AS (SELECT ... FROM b)
SELECT * FROM c;

-- Recursive: org hierarchy
WITH RECURSIVE tree(id, name, depth) AS (
  SELECT id, name, 0 FROM emp WHERE manager_id IS NULL
  UNION ALL
  SELECT e.id, e.name, t.depth+1
  FROM emp e JOIN tree t ON e.manager_id = t.id
  WHERE t.depth < 10
)
SELECT * FROM tree ORDER BY depth;

-- Recursive: integer sequence
WITH RECURSIVE n(i) AS (
  SELECT 1
  UNION ALL
  SELECT i+1 FROM n WHERE i < 100
)
SELECT i FROM n;

-- Recursive: date sequence
WITH RECURSIVE cal(d) AS (
  SELECT DATE('2024-01-01')
  UNION ALL
  SELECT DATE(d, '+1 day') FROM cal WHERE d < '2024-12-31'
)
SELECT d FROM cal;

-- Fill gaps in time series
WITH RECURSIVE cal(d) AS (...)
SELECT c.d, COALESCE(f.value, 0)
FROM cal c LEFT JOIN facts f ON c.d = f.date;

-- CTE + Window function
WITH monthly AS (
  SELECT strftime('%Y-%m', order_date) mo, SUM(amount) rev
  FROM orders GROUP BY 1
)
SELECT mo, rev,
  SUM(rev) OVER (ORDER BY mo ROWS UNBOUNDED PRECEDING) ytd
FROM monthly;
```

## Summary Map

```
SQL CTEs — ONE-PAGE SUMMARY
────────────────────────────

SYNTAX VARIATIONS
  WITH cte AS (...)                   — standard CTE
  WITH RECURSIVE cte AS (...)         — enables self-reference
  WITH cte1 AS (...), cte2 AS (...)   — multiple, ordered definitions

SIMPLE CTE USES
  ✓ Avoid repeating subquery (referenced twice in one query)
  ✓ Break complex query into readable named steps
  ✓ Compute window functions on pre-aggregated data
  ✓ Intermediate filtering before window operations

RECURSIVE CTE USES
  ✓ Org chart / tree traversal (parent → children)
  ✓ Bill of materials (item → components)
  ✓ Integer/date sequence generation
  ✓ Finding all ancestors/descendants of a node
  Always include: depth counter + WHERE depth < N (infinite loop guard)

STRUCTURE OF RECURSIVE CTE
  anchor query   — base case (seeds recursion)
  UNION ALL      — append, don't deduplicate (for performance)
  recursive step — joins back to CTE name, adds new rows
  terminates     — when recursive step returns 0 rows

INTERVIEW SIGNALS
  ✓ Know CTE vs subquery vs temp table tradeoffs
  ✓ Recursive CTE for hierarchy is a must-know pattern
  ✓ Fill missing dates = calendar CTE + LEFT JOIN
  ✓ UNION ALL (not UNION) in recursive step for performance
  ✓ Always add depth guard on recursive CTEs
```